In [ ]:
# load dataset
import pandas as pd

database_name = "smartmaldb_graph.csv"
url = (
    f"https://huggingface.co/datasets/Adson59/smartmaldb"
    f"/resolve/main/{database_name}"
)
df = pd.read_csv(url)

In [12]:
print(60*'*')
df.info()

print(60*'*')
df['label_encoded'].unique()

************************************************************
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17160 entries, 0 to 17159
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   code           17160 non-null  object 
 1   bytecode       17160 non-null  object 
 2   opcode         17160 non-null  object 
 3   label          17160 non-null  object 
 4   label_encoded  17160 non-null  int64  
 5   cfg            17160 non-null  object 
 6   adj_matrix     17160 non-null  object 
 7   cfg_dot        17160 non-null  object 
 8   cfg_nodes      17160 non-null  int64  
 9   cfg_edges      17160 non-null  int64  
 10  cfg_density    17160 non-null  float64
dtypes: float64(1), int64(3), object(7)
memory usage: 1.4+ MB
************************************************************


array([4, 3, 2, 1, 0, 5])

In [ ]:
print(60*'*')
df.shape
print(60*'*')
df.describe

In [ ]:
label_mapping = df[['label', 'label_encoded']].drop_duplicates().sort_values('label_encoded')
label_mapping = label_mapping.set_index('label_encoded')['label'].to_dict()

# Reverse mapping (label -> label_encoded)
label_to_encoded = df.groupby('label')['label_encoded'].first().to_dict()

print("=== Label to Encoded Mapping ===")
for label, encoded in sorted(label_to_encoded.items(), key=lambda x: x[1]):
    print(f"  {label:20} -> {encoded}")

print("\n=== Encoded to Label Mapping ===")
for encoded, label in sorted(label_mapping.items()):
    print(f"  {encoded} -> {label}")

# Unique labels summary
print("\n=== Dataset Summary ===")
print(f"Total samples: {len(df)}")
print(f"Unique labels: {df['label'].nunique()}")
print("\nLabel distribution:")
print(df['label'].value_counts())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.utils import resample

def create_balanced_dataset(df, chosen_label_encoded, rows_per_class, random_state=42):
    """
    Create a balanced binary dataset with benign vs. chosen vulnerability class.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The original dataset
    chosen_label_encoded : int
        The label_encoded value of the vulnerability to detect (e.g., 3 for reentrancy)
    rows_per_class : int
        Number of samples per class (benign + vulnerable)
    random_state : int
        Random seed for reproducibility
    
    Returns:
    --------
    pandas.DataFrame
        Balanced dataset with 50/50 distribution
    """
    
    # Get label mapping for reference
    label_mapping = df.groupby('label')['label_encoded'].first()
    chosen_label_name = label_mapping[label_mapping == chosen_label_encoded].index[0]
    
    print(f"Creating balanced dataset:")
    print(f"  - Benign (label_encoded=0)")
    print(f"  - Vulnerable: {chosen_label_name} (label_encoded={chosen_label_encoded})")
    print(f"  - Rows per class: {rows_per_class}")
    
    # Separate the two classes
    benign_df = df[df['label_encoded'] == 0]
    vulnerable_df = df[df['label_encoded'] == chosen_label_encoded]
    
    print(f"\nOriginal dataset:")
    print(f"  - Benign available: {len(benign_df)}")
    print(f"  - Vulnerable available: {len(vulnerable_df)}")
    
    # Check if we have enough samples
    if len(benign_df) < rows_per_class:
        print(f"[Warning] Not enough benign samples. Using all {len(benign_df)} available.")
        rows_benign = len(benign_df)
    else:
        rows_benign = rows_per_class
        
    if len(vulnerable_df) < rows_per_class:
        print(f"[Warning] Not enough vulnerable samples. Using all {len(vulnerable_df)} available.")
        rows_vulnerable = len(vulnerable_df)
    else:
        rows_vulnerable = rows_per_class
    
    # Resample to get the desired number of samples
    # For Benign
    if rows_benign < len(benign_df):
        benign_resampled = resample(
            benign_df,
            replace=False,  # Without replacement
            n_samples=rows_benign,
            random_state=random_state
        )
    else:
        benign_resampled = benign_df
    
    # For Vulnerable
    if rows_vulnerable < len(vulnerable_df):
        vulnerable_resampled = resample(
            vulnerable_df,
            replace=False,
            n_samples=rows_vulnerable,
            random_state=random_state
        )
    else:
        # If we need more than available, use oversampling (with replacement)
        if rows_per_class > len(vulnerable_df):
            print(f"[Warning] Oversampling vulnerable class...")
            vulnerable_resampled = resample(
                vulnerable_df,
                replace=True,
                n_samples=rows_per_class,
                random_state=random_state
            )
        else:
            vulnerable_resampled = vulnerable_df
    
    # Combine and shuffle
    balanced_df = pd.concat([benign_resampled, vulnerable_resampled])
    balanced_df = balanced_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    print(f"\n[Success] Balanced dataset created!")
    print(f"  - Total samples: {len(balanced_df)}")
    print(f"  - Benign: {len(balanced_df[balanced_df['label_encoded'] == 0])}")
    print(f"  - Vulnerable ({chosen_label_name}): {len(balanced_df[balanced_df['label_encoded'] == chosen_label_encoded])}")
    
    return balanced_df

In [ ]:
print("=== Available Label Mappings ===")
label_mapping = df.groupby('label')['label_encoded'].first()
for label, encoded in label_mapping.items():
    print(f"  {encoded}: {label}")
    
print("\n" + "="*50)
    
label_encoded = 3 # (e.g., unchecked_call)
    
chosen_class = 3
rows_per_class = 2300
    
balanced_df = create_balanced_dataset(
    df=df,
    chosen_label_encoded=chosen_class,
    rows_per_class=rows_per_class,
    random_state=42
)

output_file = f"unchecked_call_database_{rows_per_class}.csv"

In [ ]:
balanced_df.to_csv(output_file, index=False)
print(f"\nDataset saved to: {output_file}")

In [ ]:
INPUT_FILE = output_file
OUTPUT_FILE = "cleaned_unchecked_call_database.csv" 

# =============================================================================
LABEL_RENAME_MAP = {
    "clean": "benign",
    "external": "unchecked_calls",
}

LABEL_ENCODING_RENAME_MAP = {
    0: 0,   # benign -> 0
    3: 1,   # unchecked_call -> 2
}

# Columns to drop (if any)
COLUMNS_TO_DROP = []

# Column renaming
COLUMN_RENAME_MAP = {
    "clean": "benign",
    "external": "unchecked_call",
    # Example: "old_label": "vulnerability_type"
}

# Data cleaning settings
REMOVE_DUPLICATES = True
HANDLE_MISSING_VALUES = True
NORMALIZE_TEXT = True
REMOVE_EMPTY_OPCODES = True
STRIP_WHITESPACE = True

print("Configuration loaded!")
print(f"Input file: {INPUT_FILE}")
print(f"Output file: {OUTPUT_FILE}")

In [ ]:
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# @title: 2. Load Dataset
# @markdown: Load the dataset and perform initial inspection.
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------

print("\n" + "="*60)
print("LOADING DATASET")
print("="*60)

try:
    df = pd.read_csv(INPUT_FILE)
    print(f"✓ Dataset loaded successfully!")
    print(f"  - Shape: {df.shape}")
    print(f"  - Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
except FileNotFoundError:
    print(f"✗ File not found: {INPUT_FILE}")
    raise SystemExit(1)

# Display basic info
print(f"\n--- Dataset Info ---")
print(df.info())

print(f"\n--- First 5 Rows ---")
display(df.head())

print(f"\n--- Column Names ---")
print(list(df.columns))

print(f"\n--- Data Types ---")
print(df.dtypes)

In [21]:
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# @title: 3. Check Initial Data Quality
# @markdown: Analyze the current state of the dataset before cleaning.
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------

print("\n" + "="*60)
print("INITIAL DATA QUALITY CHECK")
print("="*60)

# 3.1 Missing Values
print("\n--- Missing Values ---")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
print(missing_df[missing_df['Missing'] > 0])

# 3.2 Duplicate Rows
print("\n--- Duplicate Rows ---")
duplicates = df.duplicated().sum()
print(f"Total duplicate rows: {duplicates}")

# 3.3 Label Distribution
print("\n--- Label Distribution (Before Cleaning) ---")
if 'label' in df.columns:
    print(df['label'].value_counts())

print("\n--- Label Encoded Distribution (Before Cleaning) ---")
if 'label_encoded' in df.columns:
    print(df['label_encoded'].value_counts().sort_index())

# 3.4 Empty/Null Analysis
print("\n--- Empty String Analysis ---")
for col in df.select_dtypes(include='object').columns:
    empty_count = (df[col] == '').sum()
    if empty_count > 0:
        print(f"  {col}: {empty_count} empty strings")



INITIAL DATA QUALITY CHECK

--- Missing Values ---
Empty DataFrame
Columns: [Missing, Percentage]
Index: []

--- Duplicate Rows ---
Total duplicate rows: 0

--- Label Distribution (Before Cleaning) ---
label
external    2300
clean       2300
Name: count, dtype: int64

--- Label Encoded Distribution (Before Cleaning) ---
label_encoded
0    2300
3    2300
Name: count, dtype: int64

--- Empty String Analysis ---


In [23]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# @title: 4. Data Cleaning Pipeline
# @markdown: Main data cleaning steps applied sequentially.
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------

print("\n" + "="*60)
print("DATA CLEANING PIPELINE")
print("="*60)

# Store original shape
original_shape = df.shape[0]

# ----------------------------------------
# 4.1 Remove Duplicate Rows
# ----------------------------------------
if REMOVE_DUPLICATES:
    print("\n[Step 4.1] Removing Duplicate Rows...")
    before = len(df)
    df = df.drop_duplicates()
    removed = before - len(df)
    print(f"  ✓ Removed {removed} duplicate rows")

# ----------------------------------------
# 4.2 Handle Missing Values
# ----------------------------------------
if HANDLE_MISSING_VALUES:
    print("\n[Step 4.2] Handling Missing Values...")
    
    # Drop rows with missing critical values
    critical_columns = ['code', 'bytecode', 'opcode', 'label', 'label_encoded']
    before = len(df)
    
    for col in critical_columns:
        if col in df.columns:
            df = df.dropna(subset=[col])
    
    removed = before - len(df)
    print(f"  ✓ Dropped {removed} rows with missing critical values")
    
    # Fill remaining missing values
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('')
        else:
            df[col] = df[col].fillna(0)

# ----------------------------------------
# 4.3 Strip Whitespace
# ----------------------------------------
if STRIP_WHITESPACE:
    print("\n[Step 4.3] Stripping Whitespace...")
    for col in df.select_dtypes(include='object').columns:
        before = len(df[df[col].str.strip() != df[col]])
        df[col] = df[col].str.strip()
        print(f"  ✓ {col}: Stripped whitespace from {before} rows")

# ----------------------------------------
# 4.4 Normalize Text Columns
# ----------------------------------------
if NORMALIZE_TEXT:
    print("\n[Step 4.4] Normalizing Text...")
    
    # Normalize code/bytecode/opcode
    normalize_cols = ['code', 'bytecode', 'opcode']
    for col in normalize_cols:
        if col in df.columns:
            # Remove extra whitespace
            df[col] = df[col].apply(lambda x: re.sub(r'\s+', ' ', str(x)) if pd.notna(x) else x)
            # Convert to lowercase where appropriate
            if col == 'label':
                df[col] = df[col].str.lower()
    
    print(f"  ✓ Text normalization complete")

# ----------------------------------------
# 4.5 Remove Empty Opcodes
# ----------------------------------------
if REMOVE_EMPTY_OPCODES:
    print("\n[Step 4.5] Removing Empty Opcodes...")
    before = len(df)
    
    if 'opcode' in df.columns:
        df = df[df['opcode'].str.len() > 0]
        df = df[df['opcode'] != '']
        df = df[df['opcode'] != 'null']
        df = df[df['opcode'].notna()]
    
    removed = before - len(df)
    print(f"  ✓ Removed {removed} rows with empty opcodes")

# ----------------------------------------
# 4.6 Remove Outliers (CFG Density)
# ----------------------------------------
print("\n[Step 4.6] Removing Extreme Outliers...")
before = len(df)

if 'cfg_density' in df.columns:
    # Remove extreme density values (likely errors)
    df = df[df['cfg_density'] > 0]
    df = df[df['cfg_density'] < 10]  # Very high density is unrealistic
    
if 'cfg_nodes' in df.columns:
    df = df[df['cfg_nodes'] > 0]
    df = df[df['cfg_nodes'] < 10000]  # Very large CFGs are likely errors

if 'cfg_edges' in df.columns:
    df = df[df['cfg_edges'] > 0]

removed = before - len(df)
print(f"  ✓ Removed {removed} outlier rows")

# ----------------------------------------
# 4.7 Validate Label Encoding Consistency
# ----------------------------------------
print("\n[Step 4.7] Validating Label Encoding Consistency...")
if 'label' in df.columns and 'label_encoded' in df.columns:
    consistency_check = df.groupby('label')['label_encoded'].nunique()
    inconsistent = consistency_check[consistency_check > 1]
    if len(inconsistent) > 0:
        print(f"  ⚠ Found inconsistent label encodings:")
        print(inconsistent)
    else:
        print(f"  ✓ All label encodings are consistent")



DATA CLEANING PIPELINE

[Step 4.1] Removing Duplicate Rows...
  ✓ Removed 0 duplicate rows

[Step 4.2] Handling Missing Values...
  ✓ Dropped 0 rows with missing critical values

[Step 4.3] Stripping Whitespace...
  ✓ code: Stripped whitespace from 0 rows
  ✓ bytecode: Stripped whitespace from 0 rows
  ✓ opcode: Stripped whitespace from 0 rows
  ✓ label: Stripped whitespace from 0 rows
  ✓ cfg: Stripped whitespace from 0 rows
  ✓ adj_matrix: Stripped whitespace from 0 rows
  ✓ cfg_dot: Stripped whitespace from 0 rows

[Step 4.4] Normalizing Text...
  ✓ Text normalization complete

[Step 4.5] Removing Empty Opcodes...
  ✓ Removed 0 rows with empty opcodes

[Step 4.6] Removing Extreme Outliers...
  ✓ Removed 0 outlier rows

[Step 4.7] Validating Label Encoding Consistency...
  ✓ All label encodings are consistent


In [24]:
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# @title: 5. Rename Labels and Encodings
# @markdown: Apply the renaming mappings defined in configuration.
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------

print("\n" + "="*60)
print("RENAMING LABELS AND ENCODINGS")
print("="*60)

# 5.1 Rename Labels
print("\n[Step 5.1] Renaming Labels...")
if 'label' in df.columns and LABEL_RENAME_MAP:
    # Check which labels exist in the dataset
    existing_labels = set(df['label'].unique())
    defined_labels = set(LABEL_RENAME_MAP.keys())
    
    # Only rename labels that exist
    rename_dict = {k: v for k, v in LABEL_RENAME_MAP.items() if k in existing_labels}
    
    if rename_dict:
        df['label'] = df['label'].map(rename_dict)
        print(f"  ✓ Renamed labels: {rename_dict}")
    else:
        print(f"  ⚠ No labels to rename (check your LABEL_RENAME_MAP)")

# 5.2 Rename Label Encodings
print("\n[Step 5.2] Renaming Label Encodings...")
if 'label_encoded' in df.columns and LABEL_ENCODING_RENAME_MAP:
    # Map the encodings
    df['label_encoded'] = df['label_encoded'].map(LABEL_ENCODING_RENAME_MAP).fillna(df['label_encoded'])
    print(f"  ✓ Renamed encodings: {LABEL_ENCODING_RENAME_MAP}")

# 5.3 Rename Columns
print("\n[Step 5.3] Renaming Columns...")
if COLUMN_RENAME_MAP:
    df = df.rename(columns=COLUMN_RENAME_MAP)
    print(f"  ✓ Renamed columns: {COLUMN_RENAME_MAP}")

# 5.4 Drop Unnecessary Columns
print("\n[Step 5.4] Dropping Unnecessary Columns...")
if COLUMNS_TO_DROP:
    existing_to_drop = [col for col in COLUMNS_TO_DROP if col in df.columns]
    if existing_to_drop:
        df = df.drop(columns=existing_to_drop)
        print(f"  ✓ Dropped columns: {existing_to_drop}")
    else:
        print(f"  ⚠ No columns to drop (check COLUMNS_TO_DROP)")




RENAMING LABELS AND ENCODINGS

[Step 5.1] Renaming Labels...
  ✓ Renamed labels: {'clean': 'benign', 'external': 'unchecked_calls'}

[Step 5.2] Renaming Label Encodings...
  ✓ Renamed encodings: {0: 0, 3: 1}

[Step 5.3] Renaming Columns...
  ✓ Renamed columns: {'clean': 'benign', 'external': 'unchecked_call'}

[Step 5.4] Dropping Unnecessary Columns...


In [25]:
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# @title: 6. Final Data Quality Check
# @markdown: Verify the dataset after cleaning.
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------

print("\n" + "="*60)
print("FINAL DATA QUALITY CHECK")
print("="*60)

# 6.1 Missing Values
print("\n--- Missing Values (After) ---")
missing_after = df.isnull().sum()
if missing_after.sum() == 0:
    print("  ✓ No missing values!")
else:
    print(missing_after[missing_after > 0])

# 6.2 Duplicate Rows
print("\n--- Duplicate Rows (After) ---")
duplicates_after = df.duplicated().sum()
print(f"  Remaining duplicates: {duplicates_after}")

# 6.3 Label Distribution
print("\n--- Label Distribution (After) ---")
if 'label' in df.columns:
    print(df['label'].value_counts())

print("\n--- Label Encoded Distribution (After) ---")
if 'label_encoded' in df.columns:
    print(df['label_encoded'].value_counts().sort_index())

# 6.4 Final Shape
print(f"\n--- Final Shape ---")
print(f"  Original rows: {original_shape}")
print(f"  Final rows: {len(df)}")
print(f"  Rows removed: {original_shape - len(df)}")
print(f"  Columns: {len(df.columns)}")

print(f"\n--- Final Column List ---")
print(list(df.columns))




FINAL DATA QUALITY CHECK

--- Missing Values (After) ---
  ✓ No missing values!

--- Duplicate Rows (After) ---
  Remaining duplicates: 0

--- Label Distribution (After) ---
label
unchecked_calls    2300
benign             2300
Name: count, dtype: int64

--- Label Encoded Distribution (After) ---
label_encoded
0    2300
1    2300
Name: count, dtype: int64

--- Final Shape ---
  Original rows: 4600
  Final rows: 4600
  Rows removed: 0
  Columns: 11

--- Final Column List ---
['code', 'bytecode', 'opcode', 'label', 'label_encoded', 'cfg', 'adj_matrix', 'cfg_dot', 'cfg_nodes', 'cfg_edges', 'cfg_density']


In [26]:
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# @title: 7. Save Cleaned Dataset
# @markdown: Save the cleaned dataset to a CSV file.
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------

print("\n" + "="*60)
print("SAVING CLEANED DATASET")
print("="*60)

df.to_csv(OUTPUT_FILE, index=False)
print(f"✓ Saved cleaned dataset to: {OUTPUT_FILE}")
print(f"  - Total samples: {len(df)}")
print(f"  - File size: {os.path.getsize(OUTPUT_FILE) / 1024**2:.2f} MB")

# Display final sample
print(f"\n--- Final Dataset Preview ---")
display(df.head())

print("\n" + "="*60)
print("DATA CLEANING COMPLETE!")
print("="*60)


SAVING CLEANED DATASET
✓ Saved cleaned dataset to: cleaned_unchecked_call_database.csv
  - Total samples: 4600
  - File size: 807.89 MB

--- Final Dataset Preview ---


,code,bytecode,opcode,label,label_encoded,cfg,adj_matrix,cfg_dot,cfg_nodes,cfg_edges,cfg_density
0,pragma solidity ^0.4.15; // From https://githu...,608060405234801561001057600080fd5b506000806000...,PUSH1 PUSH1 MSTORE CALLVALUE DUP1 ISZERO PUSH2...,unchecked_calls,1,"{'blocks': [[{'address': 0, 'opcode': 'PUSH1',...",[[0. 1. 1. ... 0. 0. 0.]\n [0. 0. 1. ... 0. 0....,digraph CFG {\nrankdir=UD;\nnode [shape=box];\...,503,636,1.264414
1,pragma solidity 0.4.15; /// @title Multisignat...,606060405234156200001057600080fd5b604051620023...,PUSH1 PUSH1 MSTORE CALLVALUE ISZERO PUSH3 JUMP...,unchecked_calls,1,"{'blocks': [[{'address': 0, 'opcode': 'PUSH1',...",[[0. 1. 1. ... 0. 0. 0.]\n [0. 0. 1. ... 0. 0....,digraph CFG {\nrankdir=UD;\nnode [shape=box];\...,644,791,1.228261
2,pragma solidity ^0.4.11; /// @title Multisigna...,60806040523480156200001157600080fd5b5060405162...,PUSH1 PUSH1 MSTORE CALLVALUE DUP1 ISZERO PUSH3...,unchecked_calls,1,"{'blocks': [[{'address': 0, 'opcode': 'PUSH1',...",[[0. 1. 1. ... 0. 0. 0.]\n [0. 0. 1. ... 0. 0....,digraph CFG {\nrankdir=UD;\nnode [shape=box];\...,559,705,1.261181
3,pragma solidity ^0.4.16; contract ARWToken { s...,606060405236156100ad576000357c0100000000000000...,PUSH1 PUSH1 MSTORE CALLDATASIZE ISZERO PUSH2 J...,benign,0,"{'blocks': [[{'address': 0, 'opcode': 'PUSH1',...",[[0. 1. 1. ... 0. 0. 0.]\n [0. 0. 1. ... 0. 0....,digraph CFG {\nrankdir=UD;\nnode [shape=box];\...,190,230,1.210526
4,pragma solidity ^0.4.10; contract EtherGame { ...,60606040526000357c0100000000000000000000000000...,PUSH1 PUSH1 MSTORE PUSH1 CALLDATALOAD PUSH29 S...,benign,0,"{'blocks': [[{'address': 0, 'opcode': 'PUSH1',...",[[0. 1. 1. ... 0. 0. 0.]\n [0. 0. 1. ... 0. 0....,digraph CFG {\nrankdir=UD;\nnode [shape=box];\...,67,77,1.149254



DATA CLEANING COMPLETE!
